# DGMAE Table 4 ablation

Roman-empire, Actor, Citeseer and Amazon Photo. The four configurations cover full DGMAE, no discrepancy selection, no feature reconstruction and no discrepancy reconstruction.

The notebook opens in `configure` mode. To check the runtime, select `smoke` for one pretraining epoch and two linear-probe epochs. Select `full` and choose a protocol for the complete experiment. The full matrix uses 10 repetitions per dataset and variant. Choose a subset of datasets, variants or seeds to fit an individual GPU session.

The [paper](https://arxiv.org/abs/2506.19343) and [public code](https://github.com/zhengziyu77/DGMAE) disagree on several settings. The `paper` mode follows Eqs. 1–13 and Appendix C. The `upstream` mode uses the public training recipe. Full `paper` runs on Roman require a split choice. The public masks use 50/25/25, and Table 7 specifies 48/32/20.

For a Roman comparison with v12, use `paper`, `paper-ratio`, `DATASETS = ["roman"]`, `VARIANTS = ["full"]` and `DISCREPANCY_NEIGHBORHOOD = "one-hop-two-hop"`. This option averages the 1-hop and 2-hop discrepancy losses. Appendix D.2 leaves the combination rule unspecified.

The preselected diagnostic uses Amazon Photo, `no-selection`, seed 2 and the same 1-hop plus 2-hop discrepancy loss as the original Photo run. `PROBE_EPOCHS` evaluates frozen embeddings from one pretraining run with the same split and probe seed. Each probe selects its own checkpoint by validation accuracy. An epoch above 1500 extends the cosine schedule across the full run, so its earlier milestones also use that extended schedule. Set `PROBE_EPOCHS = []` for a regular Table 4 run.

Use a Tesla T4 accelerator and enable internet access for the pinned checkouts, dependencies and datasets.

In [ ]:
MODE = "configure"  # "configure", "smoke" or "full"
PROTOCOL = "paper"  # "paper" or "upstream". Smoke defaults to "paper".
ROMAN_SPLIT = "paper-ratio"  # "public" or "paper-ratio". Required for a full paper run on Roman.
DISCREPANCY_NEIGHBORHOOD = "one-hop-two-hop"  # Paper only. None defaults to "one-hop".
DATASETS = ["photo"]
VARIANTS = ["no-selection"]
SEEDS = [2]  # Empty uses seed 0 for smoke or seeds 0–9 for full.
PROBE_EPOCHS = [600, 1500, 5000]  # Smoke substitutes [1].


In [ ]:
import os
from pathlib import Path
import shutil
import subprocess
import sys

assert MODE in {"configure", "smoke", "full"}, MODE
assert PROTOCOL in {None, "paper", "upstream"}, PROTOCOL
assert ROMAN_SPLIT in {None, "public", "paper-ratio"}, ROMAN_SPLIT
assert DISCREPANCY_NEIGHBORHOOD in {None, "one-hop", "one-hop-two-hop"}, DISCREPANCY_NEIGHBORHOOD
if MODE == "full" and PROTOCOL is None:
    raise ValueError("Choose PROTOCOL for a full run.")
if MODE == "full" and PROTOCOL == "paper" and "roman" in DATASETS and ROMAN_SPLIT is None:
    raise ValueError("Choose ROMAN_SPLIT for a full Roman paper run.")
if PROTOCOL == "upstream" and ROMAN_SPLIT == "paper-ratio":
    raise ValueError("Choose the public Roman split for the upstream protocol.")
if PROTOCOL == "upstream" and DISCREPANCY_NEIGHBORHOOD is not None:
    raise ValueError("DISCREPANCY_NEIGHBORHOOD requires the paper protocol.")

workdir = Path("/kaggle/working")
# Kaggle saves the results directory as output. Keep setup files in session-local /tmp.
setup_dir = Path("/tmp/dgmae-table4")
repo = setup_dir / "cs313"
upstream = setup_dir / "upstream"
experiment_dir = repo / "experiments" / "ablation"
experiment_commit = "73ae4de49b7cb6df27155385fb95c0823b7c5f62"
upstream_commit = "0e770864b5283b01e2597d6f092e8a812aa5031a"

def checkout(url, commit, destination):
    destination.mkdir(parents=True, exist_ok=True)
    subprocess.run(["git", "init", "-q", str(destination)], check=True)
    subprocess.run(["git", "-C", str(destination), "fetch", "--depth=1", url, commit], check=True)
    subprocess.run(["git", "-C", str(destination), "checkout", "--detach", commit], check=True)
    actual = subprocess.check_output(["git", "-C", str(destination), "rev-parse", "HEAD"], text=True).strip()
    if actual != commit:
        raise RuntimeError(f"Expected commit {commit}, found {actual}")

if MODE == "configure":
    print("Set MODE to smoke or full to enable execution.")
else:
    subprocess.run(["nvidia-smi", "-L"], check=True)
    checkout("https://github.com/phud774/CS313.R11.git", experiment_commit, repo)
    checkout("https://github.com/zhengziyu77/DGMAE.git", upstream_commit, upstream)
    if shutil.which("uv") is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "uv==0.13.7"], check=True)
    subprocess.run(["bash", str(experiment_dir / "setup_kaggle.sh"), str(setup_dir)], check=True)


In [ ]:
if MODE != "configure":
    launcher = experiment_dir / "run_kaggle_python.sh"
    output = workdir / "dgmae-ablation"
    env = dict(os.environ, DGL_DOWNLOAD_DIR=str(setup_dir / "datasets"), PYTHONUNBUFFERED="1")
    command = [
        "bash", str(launcher), str(setup_dir), "-B", str(experiment_dir / "run_ablation.py"),
        "--upstream", str(upstream), "--output", str(output),
        "--device", "cuda:0", "--dataset", *DATASETS, "--variant", *VARIANTS,
    ]
    if MODE == "smoke":
        command += ["--smoke"]
    if PROTOCOL is not None:
        command += ["--protocol", PROTOCOL]
    if ROMAN_SPLIT is not None:
        command += ["--roman-split", ROMAN_SPLIT]
    if DISCREPANCY_NEIGHBORHOOD is not None:
        command += ["--discrepancy-neighborhood", DISCREPANCY_NEIGHBORHOOD]
    if SEEDS:
        command += ["--seeds", *map(str, SEEDS)]
    probe_epochs = sorted(set(PROBE_EPOCHS))
    if probe_epochs:
        probe_epochs = [1] if MODE == "smoke" else probe_epochs
        command += ["--probe-epochs", *map(str, probe_epochs)]
    subprocess.run(command, env=env, check=True)


In [ ]:
if MODE != "configure":
    import json
    import pandas as pd

    protocol = PROTOCOL or "paper"
    result_dir = output / protocol / MODE
    if DISCREPANCY_NEIGHBORHOOD == "one-hop-two-hop":
        result_dir /= DISCREPANCY_NEIGHBORHOOD
    if probe_epochs:
        result_dir /= "probe-epochs-" + "-".join(map(str, probe_epochs))
        rows = []
        for path in sorted(result_dir.glob("*/*/*/seed-*.json")):
            record = json.loads(path.read_text())
            for epoch in record["epoch_probes"]:
                for split in epoch["splits"]:
                    rows.append({
                        "dataset": path.parents[2].name,
                        "variant": path.parent.name,
                        "seed": record["seed"],
                        "pretrain_epoch": epoch["epoch"],
                        "probe_epoch": split["probe_epoch"],
                        "split": split["split"],
                        "validation_accuracy_pct": 100 * split["validation_accuracy"],
                        "test_accuracy_pct": 100 * split["test_accuracy"],
                        "pretrain_discrepancy_loss": epoch["pretrain_loss"]["discrepancy"],
                    })
        if rows:
            table = pd.DataFrame(rows)
            display(table)
            table.to_csv(result_dir / "epoch_probes.csv", index=False)

    summaries = [
        json.loads(path.read_text())
        for path in sorted(result_dir.glob("*/*/*/summary.json"))
    ]
    if summaries:
        display(pd.DataFrame(summaries))
        print(f"Per-seed JSON and summaries: {result_dir}")
